In [3]:
# !pip install -q langchain langchain-community langchain-text-splitters chromadb sentence-transformers beautifulsoup4 requests lxml

In [4]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

c:\Users\mallo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Document list

In [5]:
documents_info = [
    # --- Original 20 ---
    {"topic": "Chest Pain", "source": "MedlinePlus", "url": "https://medlineplus.gov/ency/article/003079.htm", "date": "2024-05-08"},
    {"topic": "Fever", "source": "MedlinePlus", "url": "https://medlineplus.gov/fever.html", "date": "2025-07-27"},
    {"topic": "Headache", "source": "WHO", "url": "https://www.who.int/news-room/fact-sheets/detail/headache-disorders", "date": "2025-10-24"},
    {"topic": "Cough", "source": "NHS", "url": "https://www.nhs.uk/symptoms/cough/", "date": "2023-12-08"},
    {"topic": "Shortness of Breath", "source": "NHS", "url": "https://www.nhs.uk/conditions/shortness-of-breath/", "date": "2024-01-30"},
    {"topic": "Abdominal Pain", "source": "MedlinePlus", "url": "https://medlineplus.gov/abdominalpain.html", "date": "2025-09-23"},
    {"topic": "Vomiting and Diarrhea", "source": "NHS", "url": "https://www.nhs.uk/conditions/diarrhoea-and-vomiting/", "date": "2023-12-21"},
    {"topic": "Sore Throat", "source": "CDC", "url": "https://www.cdc.gov/sore-throat/about/index.html", "date": "2024-04-17"},
    {"topic": "Burns and Scalds", "source": "NHS", "url": "https://www.nhs.uk/conditions/burns-and-scalds/", "date": "2022-06-23"},
    {"topic": "Cuts and Scrapes", "source": "MedlinePlus", "url": "https://medlineplus.gov/firstaid.html", "date": "2026-01-25"},
    {"topic": "Sprains", "source": "NHS", "url": "https://www.nhs.uk/conditions/sprains-and-strains/", "date": "2024-04-23"},
    {"topic": "Allergic Reaction (Anaphylaxis)", "source": "MedlinePlus", "url": "https://medlineplus.gov/ency/article/000844.htm", "date": "2024-03-31"},
    {"topic": "Fever in Infants", "source": "NHS", "url": "https://www.nhs.uk/conditions/fever-in-children/", "date": "2024-01-03"},
    {"topic": "Stroke Symptoms", "source": "CDC", "url": "https://www.cdc.gov/stroke/signs-symptoms/", "date": "2024-10-24"},
    {"topic": "Sepsis", "source": "WHO", "url": "https://www.who.int/news-room/fact-sheets/detail/sepsis", "date": "2024-05-03"},
    {"topic": "Meningitis", "source": "NHS", "url": "https://www.nhs.uk/conditions/meningitis/", "date": "2026-03-18"},
    {"topic": "Deep Vein Thrombosis (DVT)", "source": "CDC", "url": "https://www.cdc.gov/blood-clots/about/", "date": "2025-03-05"},
    {"topic": "Dehydration", "source": "NHS", "url": "https://www.nhs.uk/conditions/dehydration/", "date": "2022-11-14"},
    {"topic": "Signs of a Panic Attack", "source": "NIMH", "url": "https://www.nimh.nih.gov/health/publications/panic-disorder-when-fear-overwhelms", "date": "2025"},
    {"topic": "High Blood Pressure in Pregnancy", "source": "MedlinePlus", "url": "https://medlineplus.gov/highbloodpressureinpregnancy.html", "date": "2024-05-29"},

    # --- Extra 70 (from extra_docs.pdf, entries 21–90) ---
    {"topic": "Common Cold", "source": "MedlinePlus", "url": "https://medlineplus.gov/commoncold.html", "date": "2022-11-16"},
    {"topic": "Constipation", "source": "MedlinePlus", "url": "https://medlineplus.gov/constipation.html", "date": "2025-07-21"},
    {"topic": "Motion Sickness", "source": "MedlinePlus", "url": "https://medlineplus.gov/motionsickness.html", "date": "2024-09-23"},
    {"topic": "Sunburn", "source": "MedlinePlus", "url": "https://medlineplus.gov/ency/article/003227.htm", "date": "2025-02-11"},
    {"topic": "Hay Fever (Seasonal Allergies)", "source": "MedlinePlus", "url": "https://medlineplus.gov/hayfever.html", "date": "2018-06-28"},
    {"topic": "Mouth Ulcers", "source": "NHS", "url": "https://www.nhs.uk/conditions/mouth-ulcers/", "date": "2024-03-11"},
    {"topic": "Cold Sores", "source": "NHS", "url": "https://www.nhs.uk/conditions/cold-sores/", "date": "2024-02-19"},
    {"topic": "Insect Bites and Stings", "source": "NHS", "url": "https://www.nhs.uk/conditions/insect-bites-and-stings/", "date": "2023-06-01"},
    {"topic": "Dandruff", "source": "NHS", "url": "https://www.nhs.uk/conditions/dandruff/", "date": "2022-12-23"},
    {"topic": "Athlete's Foot", "source": "NHS", "url": "https://www.nhs.uk/conditions/athletes-foot/", "date": "2024-04-29"},
    {"topic": "Ringworm", "source": "NHS", "url": "https://www.nhs.uk/conditions/ringworm/", "date": "2023-08-03"},
    {"topic": "Warts and Verrucas", "source": "NHS", "url": "https://www.nhs.uk/conditions/warts-and-verrucas/", "date": "2023-07-25"},
    {"topic": "Mild Dry Skin", "source": "NHS", "url": "https://www.shropshiretelfordandwrekin.nhs.uk/self-care/mild-dry-skin/", "date": "2022-06-29"},
    {"topic": "Earache", "source": "NHS", "url": "https://www.nhs.uk/symptoms/earache/", "date": "2025-10-27"},
    {"topic": "Stye", "source": "NHS", "url": "https://www.nhs.uk/conditions/stye/", "date": "2024-04-18"},
    {"topic": "Atopic Eczema", "source": "NHS", "url": "https://www.nhs.uk/conditions/atopic-eczema/", "date": "2024-09-06"},
    {"topic": "Acne", "source": "MedlinePlus", "url": "https://medlineplus.gov/acne.html", "date": "2024-10-12"},
    {"topic": "Insomnia", "source": "MedlinePlus", "url": "https://medlineplus.gov/insomnia.html", "date": "2024-09-13"},
    {"topic": "Pink Eye (Conjunctivitis)", "source": "CDC", "url": "https://www.cdc.gov/conjunctivitis/about/index.html", "date": "2024-04-15"},
    {"topic": "Contact Dermatitis", "source": "NHS", "url": "https://www.nhs.uk/conditions/contact-dermatitis/", "date": "2023-05-03"},
    {"topic": "Nosebleed", "source": "NHS", "url": "https://www.nhs.uk/conditions/nosebleed/", "date": "2023-12-05"},
    {"topic": "Blisters", "source": "NHS", "url": "https://www.nhs.uk/conditions/blisters/", "date": "2023-11-22"},
    {"topic": "Piles (Haemorrhoids)", "source": "NHS", "url": "https://www.nhs.uk/conditions/piles-haemorrhoids/", "date": "2026-04-01"},
    {"topic": "Sinusitis (Sinus Infection)", "source": "NHS", "url": "https://www.nhs.uk/conditions/sinusitis-sinus-infection/", "date": "2024-01-31"},
    {"topic": "Ear Infections", "source": "NHS", "url": "https://www.nhs.uk/conditions/ear-infections/", "date": "2025-01-16"},
    {"topic": "Back Pain", "source": "NHS", "url": "https://www.nhs.uk/conditions/back-pain/", "date": "2026-03-05"},
    {"topic": "Sciatica", "source": "NHS", "url": "https://www.nhs.uk/conditions/sciatica/", "date": "2024-12-03"},
    {"topic": "Migraine", "source": "MedlinePlus", "url": "https://medlineplus.gov/migraine.html", "date": "2025-11-20"},
    {"topic": "Shingles", "source": "NHS", "url": "https://www.nhs.uk/conditions/shingles/", "date": "2023-11-23"},
    {"topic": "Heartburn", "source": "MedlinePlus", "url": "https://medlineplus.gov/heartburn.html", "date": "2024-09-17"},
    {"topic": "Urinary Tract Infection (UTI)", "source": "NHS", "url": "https://www.nhs.uk/conditions/urinary-tract-infections-utis/", "date": "2025-07-11"},
    {"topic": "Indigestion", "source": "NHS", "url": "https://www.nhs.uk/conditions/indigestion/", "date": "2023-05-05"},
    {"topic": "Psoriasis", "source": "NHS", "url": "https://www.nhs.uk/conditions/psoriasis/", "date": "2026-03-10"},
    {"topic": "Thrush in Men and Women", "source": "NHS", "url": "https://www.nhs.uk/conditions/thrush-in-men-and-women/", "date": "2023-07-28"},
    {"topic": "Bacterial Vaginosis", "source": "NHS", "url": "https://www.nhs.uk/conditions/bacterial-vaginosis/", "date": "2022-10-27"},
    {"topic": "Toothache", "source": "NHS", "url": "https://www.nhs.uk/conditions/toothache/", "date": "2024-07-01"},
    {"topic": "Ingrown Toenail", "source": "NHS", "url": "https://www.nhs.uk/conditions/ingrown-toenail/", "date": "2025-06-30"},
    {"topic": "Tinnitus", "source": "NHS", "url": "https://www.nhs.uk/conditions/tinnitus/", "date": "2024-01-12"},
    {"topic": "Hearing Loss", "source": "NHS", "url": "https://www.nhs.uk/conditions/hearing-loss/", "date": "2025-05-30"},
    {"topic": "GERD / Acid Reflux", "source": "MedlinePlus", "url": "https://medlineplus.gov/gerd.html", "date": "2025-11-24"},
    {"topic": "Irritable Bowel Syndrome (IBS)", "source": "NHS", "url": "https://www.nhs.uk/conditions/irritable-bowel-syndrome-ibs/symptoms/", "date": "2025-03-17"},
    {"topic": "Kidney Stones", "source": "NHS", "url": "https://www.nhs.uk/conditions/kidney-stones/", "date": "2022-11-30"},
    {"topic": "Blood in Urine", "source": "NHS", "url": "https://www.nhs.uk/conditions/blood-in-urine/", "date": "2023-05-19"},
    {"topic": "Swollen Glands", "source": "NHS", "url": "https://www.nhs.uk/conditions/swollen-glands/", "date": "2023-09-29"},
    {"topic": "Depression", "source": "NIMH", "url": "https://www.nimh.nih.gov/health/topics/depression", "date": "2024-12"},
    {"topic": "Anxiety Disorders", "source": "NIMH", "url": "https://www.nimh.nih.gov/health/topics/anxiety-disorders", "date": "2024-12"},
    {"topic": "Endometriosis", "source": "NHS", "url": "https://www.nhs.uk/conditions/endometriosis/", "date": "2024-08-27"},
    {"topic": "Heavy Periods", "source": "NHS", "url": "https://www.nhs.uk/conditions/heavy-periods/", "date": "2024-09-19"},
    {"topic": "Gallstones", "source": "NHS", "url": "https://www.nhs.uk/conditions/gallstones/", "date": "2025-08-11"},
    {"topic": "Acne (Self-care)", "source": "MedlinePlus", "url": "https://medlineplus.gov/ency/patientinstructions/000750.htm", "date": "2024-02-15"},
    {"topic": "Eczema", "source": "MedlinePlus", "url": "https://medlineplus.gov/eczema.html", "date": "2025-06-15"},
    {"topic": "Appendicitis", "source": "NHS", "url": "https://www.nhs.uk/conditions/appendicitis/", "date": "2024-08-09"},
    {"topic": "Carbon Monoxide Poisoning", "source": "CDC", "url": "https://www.cdc.gov/carbon-monoxide/about/index.html", "date": "2026-01-12"},
    {"topic": "Pulmonary Embolism", "source": "MedlinePlus", "url": "https://medlineplus.gov/pulmonaryembolism.html", "date": "2024-01-26"},
    {"topic": "Ectopic Pregnancy", "source": "NHS", "url": "https://www.nhs.uk/conditions/ectopic-pregnancy/", "date": "2022-08-23"},
    {"topic": "Asthma Attack", "source": "NHS", "url": "https://www.nhs.uk/conditions/asthma/asthma-attack/", "date": "2025-04-07"},
    {"topic": "Head Injury / Concussion", "source": "CDC", "url": "https://www.cdc.gov/traumatic-brain-injury/signs-symptoms/index.html", "date": "2025-09-15"},
    {"topic": "Seizures", "source": "MedlinePlus", "url": "https://medlineplus.gov/seizures.html", "date": "2017-06-29"},
    {"topic": "Poisoning", "source": "MedlinePlus", "url": "https://medlineplus.gov/poisoning.html", "date": "2018-07-12"},
    {"topic": "Heat Stroke / Extreme Heat Illness", "source": "CDC", "url": "https://www.cdc.gov/extreme-heat/signs-symptoms/index.html", "date": "2025-07-25"},
    {"topic": "Diabetic Ketoacidosis (DKA)", "source": "MedlinePlus", "url": "https://medlineplus.gov/ency/article/000320.htm", "date": "2025-01-10"},
    {"topic": "Low Blood Glucose (Hypoglycemia)", "source": "NIDDK", "url": "https://www.niddk.nih.gov/health-information/diabetes/overview/preventing-problems/low-blood-glucose-hypoglycemia", "date": "2021-07"},
    {"topic": "Gastrointestinal Bleeding", "source": "MedlinePlus", "url": "https://medlineplus.gov/gastrointestinalbleeding.html", "date": "2024-11-13"},
    {"topic": "Fainting (Syncope)", "source": "MedlinePlus", "url": "https://medlineplus.gov/fainting.html", "date": "2025-07-22"},
    {"topic": "Sudden Vision Loss", "source": "NHS", "url": "https://www.nhs.uk/conditions/vision-loss/", "date": "2025-08-28"},
    {"topic": "Kidney Infection", "source": "NHS", "url": "https://www.nhs.uk/conditions/kidney-infection/", "date": "2025-02-13"},
    {"topic": "Pneumonia", "source": "NHS", "url": "https://www.nhs.uk/conditions/pneumonia/", "date": "2023-01-12"},
    {"topic": "Testicular Torsion", "source": "Cleveland Clinic", "url": "https://my.clevelandclinic.org/health/diseases/15382-testicular-torsion", "date": "2023-02-27"},
    {"topic": "Retinal Detachment", "source": "NHS", "url": "https://www.nhs.uk/conditions/retinal-detachment/", "date": "2023-08-04"},
    {"topic": "Suicidal Thoughts / Crisis Support", "source": "NIMH", "url": "https://www.nimh.nih.gov/health/topics/suicide-prevention", "date": "2025-08"},
]

cleaning the text from weird characters 

In [6]:
def clean_text(text):
    replacements = {
        "â": "'",
        "â": '"',
        "â": '"',
        "â": "-",
        "â": "-",
        "Â": " ",
        "\xa0": " ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Extract sections from each webpage

This is important.
Instead of taking one huge text block, we extract by headings so we can later store in metadata.

# Phase 1: ONLY HTML cleaning

In [7]:
def extract_sections_from_url(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    # Remove noisy parts (FILTER 1)
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.extract()

    main_content = None
    for selector in ["main", "article", "[role='main']"]:
        main_content = soup.select_one(selector)
        if main_content:
            break

    container = main_content if main_content else soup

    sections = []
    current_title = "Introduction"
    current_text = []

    for element in container.find_all(["h1", "h2", "h3", "p", "li"]):
        if element.name in ["h1", "h2", "h3"]:
            if current_text:
                joined_text = clean_text(" ".join(current_text))
                if joined_text:
                    sections.append({
                        "section_title": current_title,
                        "text": joined_text
                    })
                current_text = []

            heading = clean_text(element.get_text(" ", strip=True))
            current_title = heading if heading else "Untitled Section"

        elif element.name in ["p", "li"]:
            txt = clean_text(element.get_text(" ", strip=True))
            if txt:
                current_text.append(txt)

    if current_text:
        joined_text = clean_text(" ".join(current_text))
        if joined_text:
            sections.append({
                "section_title": current_title,
                "text": joined_text
            })

    return sections

Scrape all webpages: This builds raw_docs.

In [8]:
raw_docs_phase1 = []

for i, item in enumerate(documents_info, start=1):
    try:
        print(f"[Phase 1] Processing {i}: {item['topic']}")
        sections = extract_sections_from_url(item["url"])

        raw_docs_phase1.append({
            "sections": sections,
            "metadata": item
        })

        print(f"Sections extracted: {len(sections)}")
        time.sleep(1)

    except Exception as e:
        print(f"Failed on {item['topic']}: {e}")

[Phase 1] Processing 1: Chest Pain
Sections extracted: 12
[Phase 1] Processing 2: Fever
Sections extracted: 18
[Phase 1] Processing 3: Headache
Sections extracted: 12
[Phase 1] Processing 4: Cough
Sections extracted: 9
[Phase 1] Processing 5: Shortness of Breath
Sections extracted: 7
[Phase 1] Processing 6: Abdominal Pain
Sections extracted: 15
[Phase 1] Processing 7: Vomiting and Diarrhea
Sections extracted: 10
[Phase 1] Processing 8: Sore Throat
Sections extracted: 11
[Phase 1] Processing 9: Burns and Scalds
Sections extracted: 9
[Phase 1] Processing 10: Cuts and Scrapes
Sections extracted: 16
[Phase 1] Processing 11: Sprains
Sections extracted: 11
[Phase 1] Processing 12: Allergic Reaction (Anaphylaxis)
Sections extracted: 14
[Phase 1] Processing 13: Fever in Infants
Sections extracted: 9
[Phase 1] Processing 14: Stroke Symptoms
Sections extracted: 7
[Phase 1] Processing 15: Sepsis
Sections extracted: 8
[Phase 1] Processing 16: Meningitis
Sections extracted: 8
[Phase 1] Processing 1

Visualize the scraped sections

In [9]:
for doc in raw_docs_phase1[0:1]:
    print("\n" + "=" * 100)
    print("DOCUMENT METADATA")
    print("-" * 40)

    for key, value in doc["metadata"].items():
        print(f"{key:<15}: {value}")

    print("\nTOTAL SECTIONS:", len(doc["sections"]))
    print("-" * 40)

    for i, sec in enumerate(doc["sections"][:5], start=1):
        print(f"\n[SECTION {i}]")
        print(f"Title   : {sec['section_title']}")
        print(f"Chars   : {len(sec['text'])}")
        print(f"Preview : {sec['text'][:250]}...")


DOCUMENT METADATA
----------------------------------------
topic          : Chest Pain
source         : MedlinePlus
url            : https://medlineplus.gov/ency/article/003079.htm
date           : 2024-05-08

TOTAL SECTIONS: 12
----------------------------------------

[SECTION 1]
Title   : Chest pain
Chars   : 121
Preview : Chest pain is discomfort or pain that you feel anywhere along the front of your body between your neck and upper abdomen....

[SECTION 2]
Title   : Considerations
Chars   : 516
Preview : Many people with chest pain fear that they are having a heart attack (myocardial infarction). However, there are many possible causes of chest pain. Some causes are not dangerous to your health, while other causes are serious and, in some cases, life...

[SECTION 3]
Title   : Causes
Chars   : 1742
Preview : Heart or blood vessel problems that can cause chest pain: Angina or a heart attack. The most common symptom is chest pain that may feel like tightness, heavy pressure, squeezi

# Phase 2 — Add length filter (Filter 1 + Filter 2)
Clean the sections to remove navigation/UI content etc.. 

In [10]:
MIN_CHARS = 100
MIN_WORDS = 20

raw_docs_phase2 = []

for i, item in enumerate(documents_info, start=1):
    try:
        print(f"[Phase 2] Processing {i}: {item['topic']}")

        sections = extract_sections_from_url(item["url"])

        clean_sections = []
        removed_sections = []

        for sec in sections:
            text = sec["text"]

            if len(text) >= MIN_CHARS and len(text.split()) >= MIN_WORDS:
                clean_sections.append(sec)
            else:
                removed_sections.append(sec)

        raw_docs_phase2.append({
            "sections": clean_sections,
            "removed_sections": removed_sections,   # for debugging
            "metadata": item
        })

        print(
            f"Original: {len(sections)} | "
            f"Kept: {len(clean_sections)} | "
            f"Removed: {len(removed_sections)}"
        )

        time.sleep(1)

    except Exception as e:
        print(f"Failed on {item['topic']}: {e}")

[Phase 2] Processing 1: Chest Pain
Original: 12 | Kept: 9 | Removed: 3
[Phase 2] Processing 2: Fever
Original: 18 | Kept: 8 | Removed: 10
[Phase 2] Processing 3: Headache
Original: 12 | Kept: 12 | Removed: 0
[Phase 2] Processing 4: Cough
Original: 9 | Kept: 7 | Removed: 2
[Phase 2] Processing 5: Shortness of Breath
Original: 7 | Kept: 7 | Removed: 0
[Phase 2] Processing 6: Abdominal Pain
Original: 15 | Kept: 6 | Removed: 9
[Phase 2] Processing 7: Vomiting and Diarrhea
Original: 10 | Kept: 10 | Removed: 0
[Phase 2] Processing 8: Sore Throat
Original: 11 | Kept: 11 | Removed: 0
[Phase 2] Processing 9: Burns and Scalds
Original: 9 | Kept: 8 | Removed: 1
[Phase 2] Processing 10: Cuts and Scrapes
Original: 16 | Kept: 6 | Removed: 10
[Phase 2] Processing 11: Sprains
Original: 11 | Kept: 11 | Removed: 0
[Phase 2] Processing 12: Allergic Reaction (Anaphylaxis)
Original: 14 | Kept: 10 | Removed: 4
[Phase 2] Processing 13: Fever in Infants
Original: 9 | Kept: 9 | Removed: 0
[Phase 2] Processing 

In [11]:
for doc in raw_docs_phase2[0:1]:
    print("\n" + "=" * 100)
    print("DOCUMENT METADATA")
    print("-" * 40)

    for key, value in doc["metadata"].items():
        print(f"{key:<15}: {value}")

    print("\nKEPT SECTIONS:", len(doc["sections"]))
    print("-" * 40)

    for i, sec in enumerate(doc["sections"][:5], start=1):
        print(f"\n[KEPT SECTION {i}]")
        print(f"Title   : {sec['section_title']}")
        print(f"Chars   : {len(sec['text'])}")
        print(f"Words   : {len(sec['text'].split())}")
        print(f"Preview : {sec['text'][:250]}...")


DOCUMENT METADATA
----------------------------------------
topic          : Chest Pain
source         : MedlinePlus
url            : https://medlineplus.gov/ency/article/003079.htm
date           : 2024-05-08

KEPT SECTIONS: 9
----------------------------------------

[KEPT SECTION 1]
Title   : Chest pain
Chars   : 121
Words   : 22
Preview : Chest pain is discomfort or pain that you feel anywhere along the front of your body between your neck and upper abdomen....

[KEPT SECTION 2]
Title   : Considerations
Chars   : 516
Words   : 90
Preview : Many people with chest pain fear that they are having a heart attack (myocardial infarction). However, there are many possible causes of chest pain. Some causes are not dangerous to your health, while other causes are serious and, in some cases, life...

[KEPT SECTION 3]
Title   : Causes
Chars   : 1742
Words   : 313
Preview : Heart or blood vessel problems that can cause chest pain: Angina or a heart attack. The most common symptom is chest pain 

In [12]:
for doc in raw_docs_phase2[1:2]:
    print("\n" + "=" * 100)
    print("REMOVED SECTIONS")
    print("-" * 40)

    for i, sec in enumerate(doc["removed_sections"][:5], start=1):
        print(f"\n[REMOVED {i}]")
        print(f"Title   : {sec['section_title']}")
        print(f"Chars   : {len(sec['text'])}")
        print(f"Words   : {len(sec['text'].split())}")
        print(f"Preview : {sec['text'][:200]}...")


REMOVED SECTIONS
----------------------------------------

[REMOVED 1]
Title   : Basics
Chars   : 63
Words   : 9
Preview : Summary Start Here Diagnosis and Tests Treatments and Therapies...

[REMOVED 2]
Title   : Learn More
Chars   : 18
Words   : 2
Preview : Specifics Genetics...

[REMOVED 3]
Title   : See, Play and Learn
Chars   : 18
Words   : 3
Preview : No links available...

[REMOVED 4]
Title   : Research
Chars   : 32
Words   : 4
Preview : Clinical Trials Journal Articles...

[REMOVED 5]
Title   : Resources
Chars   : 14
Words   : 3
Preview : Find an Expert...


save scraped docs

In [13]:
save_dir = os.path.join(os.getcwd(), "cleaned_docs_phase2")
os.makedirs(save_dir, exist_ok=True)

for doc in raw_docs_phase2:
    safe_topic = doc["metadata"]["topic"].replace(" ", "_").replace("/", "_")
    filepath = os.path.join(save_dir, f"{safe_topic}.txt")

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"TOPIC: {doc['metadata']['topic']}\n")
        f.write(f"SOURCE: {doc['metadata']['source']}\n")
        f.write(f"URL: {doc['metadata']['url']}\n")
        f.write(f"DATE: {doc['metadata'].get('date')}\n\n")

        for sec in doc["sections"]:
            f.write(f"SECTION TITLE: {sec['section_title']}\n")
            f.write(sec["text"] + "\n\n")

print(f"Saved cleaned text files in: {save_dir}")

Saved cleaned text files in: c:\Users\mallo\OneDrive - Qatar University\Desktop\LLMS\project\Part A-C\cleaned_docs_phase2


In [14]:
print(os.listdir("cleaned_docs_phase2"))

['Abdominal_Pain.txt', 'Acne.txt', 'Acne_(Self-care).txt', 'Allergic_Reaction_(Anaphylaxis).txt', 'Anxiety_Disorders.txt', 'Appendicitis.txt', 'Asthma_Attack.txt', "Athlete's_Foot.txt", 'Atopic_Eczema.txt', 'Back_Pain.txt', 'Bacterial_Vaginosis.txt', 'Blisters.txt', 'Blood_in_Urine.txt', 'Burns_and_Scalds.txt', 'Carbon_Monoxide_Poisoning.txt', 'Chest_Pain.txt', 'Cold_Sores.txt', 'Common_Cold.txt', 'Constipation.txt', 'Contact_Dermatitis.txt', 'Cough.txt', 'Cuts_and_Scrapes.txt', 'Dandruff.txt', 'Deep_Vein_Thrombosis_(DVT).txt', 'Dehydration.txt', 'Depression.txt', 'Diabetic_Ketoacidosis_(DKA).txt', 'Earache.txt', 'Ear_Infections.txt', 'Ectopic_Pregnancy.txt', 'Eczema.txt', 'Endometriosis.txt', 'Fainting_(Syncope).txt', 'Fever.txt', 'Fever_in_Infants.txt', 'Gallstones.txt', 'Gastrointestinal_Bleeding.txt', 'GERD___Acid_Reflux.txt', 'Hay_Fever_(Seasonal_Allergies).txt', 'Headache.txt', 'Head_Injury___Concussion.txt', 'Hearing_Loss.txt', 'Heartburn.txt', 'Heat_Stroke___Extreme_Heat_Illnes

json copy

In [15]:
import json

with open("cleaned_docs_phase2.json", "w", encoding="utf-8") as f:
    json.dump(raw_docs_phase2, f, indent=2, ensure_ascii=False)

print("Saved cleaned dataset as cleaned_docs_phase2.json")


Saved cleaned dataset as cleaned_docs_phase2.json


In [16]:
with open("cleaned_docs_phase2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of docs:", len(data))
print("First topic:", data[0]["metadata"]["topic"])
print("First section title:", data[0]["sections"][0]["section_title"])

Number of docs: 90
First topic: Chest Pain
First section title: Chest pain


# chunking starts now

In [17]:
import json

with open("cleaned_docs_phase2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of docs:", len(data))
print("First topic:", data[0]["metadata"]["topic"])
print("First section title:", data[0]["sections"][0]["section_title"])

Number of docs: 90
First topic: Chest Pain
First section title: Chest pain


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

# =========================================
# Tokenizer (same family as embedding model)
# =========================================
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def count_tokens(text):
    return len(tokenizer.encode(text))


# =========================================
# Token-based splitter (FIXED)
# =========================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,          # tokens
    chunk_overlap=50,        # tokens
    length_function=count_tokens,
    separators=["\n\n", "\n", ".", " ", ""]
)



threshold=15 #################################################
# =========================================
# Optional: merge very small chunks
# =========================================
def merge_small_chunks(chunks, min_words=threshold):
    merged = []
    for chunk in chunks:
        if merged and len(chunk.split()) < min_words:
            merged[-1] += " " + chunk
        else:
            merged.append(chunk)
    return merged
# =========================================
# Tracking discarded items
# =========================================
discarded_sections = []
discarded_chunks = []

discarded_section_count = 0
discarded_chunk_count = 0


# =========================================
# Chunking pipeline
# =========================================
chunked_data = []
global_chunk_id = 0

for source_doc in data:
    base = source_doc["metadata"]
    source_chunk_index = 0

    for section_idx, section in enumerate(source_doc["sections"]):
        section_text = section["text"]
        section_title = section["section_title"]

        # ---------------------------------
        # Track discarded SHORT SECTIONS
        # ---------------------------------
        if len(section_text.split()) < threshold:
            discarded_section_count += 1
            discarded_sections.append({
                "section_title": section_title,
                "source": base["source"],
                "words": len(section_text.split()),
                "text": section_text[:200]
            })
            continue

        # Split into chunks
        chunks = text_splitter.split_text(section_text)

        # Merge small chunks
        chunks = merge_small_chunks(chunks, min_words=threshold)

        for chunk_text in chunks:

            # ---------------------------------
            # Track discarded SMALL CHUNKS
            # ---------------------------------
            if len(chunk_text.split()) < threshold:
                discarded_chunk_count += 1
                discarded_chunks.append({
                    "section_title": section_title,
                    "source": base["source"],
                    "words": len(chunk_text.split()),
                    "text": chunk_text[:200]
                })
                continue

            # Keep valid chunk
            chunked_data.append({
                "page_content": chunk_text,
                "metadata": {
                    "topic": base["topic"],
                    "source": base["source"],
                    "url": base["url"],
                    "page_number": None,
                    "section_title": section_title,
                    "document_date": base.get("date"),
                    "author": None,
                    "chunk_index": source_chunk_index,
                    "global_chunk_id": global_chunk_id
                }
            })

            source_chunk_index += 1
            global_chunk_id += 1


# =========================================
# Summary prints
# =========================================
print("\n========== SUMMARY ==========")
print("Total chunks kept       :", len(chunked_data))
print("Discarded sections      :", discarded_section_count)
print("Discarded small chunks  :", discarded_chunk_count)


# =========================================
# Preview discarded sections
# =========================================
print("\n========== DISCARDED SECTIONS (sample) ==========")
for s in discarded_sections[:5]:
    print("\nSection:", s["section_title"])
    print("Source :", s["source"])
    print("Words  :", s["words"])
    print("Text   :", s["text"])


# =========================================
# Preview discarded chunks
# =========================================
print("\n========== DISCARDED CHUNKS (sample) ==========")
for c in discarded_chunks[:5]:
    print("\nSection:", c["section_title"])
    print("Source :", c["source"])
    print("Words  :", c["words"])
    print("Text   :", c["text"])

Token indices sequence length is longer than the specified maximum sequence length for this model (748 > 512). Running this sequence through the model will result in indexing errors



========== SUMMARY ==========
Total chunks kept       : 1108
Discarded sections      : 0
Discarded small chunks  : 2

========== DISCARDED SECTIONS (sample) ==========

========== DISCARDED CHUNKS (sample) ==========

Section: Recovering at home
Source : NHS
Words  : 10
Text   : There are things you can do to help your recovery

Section: Genetics
Source : MedlinePlus
Words  : 1
Text   : 15q13


In [19]:
for i, d in enumerate(chunked_data[:20], start=19):
    text = d["page_content"]
    meta = d["metadata"]

    print("\n" + "="*80)
    print(f"Chunk {i}")
    print("="*80)

    print(f"Global ID       : {meta['global_chunk_id']}")
    print(f"Chunk index     : {meta['chunk_index']}")
    print(f"Topic           : {meta['topic']}")
    print(f"Section         : {meta['section_title']}")
    print(f"Source          : {meta['source']}")
    print(f"Author          : {meta['author']}")
    print(f"Date            : {meta['document_date']}")
    print(f"URL             : {meta['url']}")


    print(f"Words           : {len(text.split())}")
    print(f"Tokens          : {count_tokens(text)}")

    print(f"\nPreview         : {text[:200]}...")


Chunk 19
Global ID       : 0
Chunk index     : 0
Topic           : Chest Pain
Section         : Chest pain
Source          : MedlinePlus
Author          : None
Date            : 2024-05-08
URL             : https://medlineplus.gov/ency/article/003079.htm
Words           : 22
Tokens          : 25

Preview         : Chest pain is discomfort or pain that you feel anywhere along the front of your body between your neck and upper abdomen....

Chunk 20
Global ID       : 1
Chunk index     : 1
Topic           : Chest Pain
Section         : Considerations
Source          : MedlinePlus
Author          : None
Date            : 2024-05-08
URL             : https://medlineplus.gov/ency/article/003079.htm
Words           : 90
Tokens          : 120

Preview         : Many people with chest pain fear that they are having a heart attack (myocardial infarction). However, there are many possible causes of chest pain. Some causes are not dangerous to your health, while...

Chunk 21
Global ID       : 2
Ch

In [20]:
with open("chunked_docs_phase2.json", "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, indent=2, ensure_ascii=False)

print("Saved chunked documents to chunked_docs_phase2.json")

Saved chunked documents to chunked_docs_phase2.json


Convert chunked data to LangChain Document objects

In [21]:
from langchain_core.documents import Document

chunked_docs = [
    Document(page_content=d["page_content"], metadata=d["metadata"])
    for d in chunked_data
]

print("Converted to LangChain Documents:", len(chunked_docs))

Converted to LangChain Documents: 1108


# create embeddings model

In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

C:\Users\mallo\AppData\Local\Temp\ipykernel_28104\1216582700.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20587.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


delete chroma before rerunning or it will make duplicates

In [23]:
import shutil
import os

persist_dir = "chroma_db_phase2"

if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("Deleted successfully")
else:
    print("Folder does not exist")

Deleted successfully


In [24]:
# #If you do not want to restart right now, use a new folder name instead. That is the quickest workaround
# from langchain_community.vectorstores import Chroma

# new_persist_dir = "chroma_db_phase2_clean"

# vectorstore = Chroma.from_documents(
#     documents=chunked_docs,
#     embedding=embedding_model,
#     persist_directory=new_persist_dir
# )

# print("New clean Chroma DB created.")

Create Chroma vector database

In [25]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embedding_model,
    persist_directory="chroma_db_phase2"
)

print("Chroma DB created and saved locally.")

Chroma DB created and saved locally.


# Retrieval 

dense similarity retrieval with configurable top-k

In [26]:
# ================================
# 1) Dense retrieval from Chroma
# ================================

def get_dense_retriever(vectorstore, k=5):
    """
    Returns a Chroma similarity retriever with configurable top-k.
    """
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )


def run_dense_search(query, vectorstore, k=5):
    """
    Runs dense similarity search and returns top-k documents.
    """
    retriever = get_dense_retriever(vectorstore, k=k)
    docs = retriever.invoke(query)
    return docs


In [27]:
query = "What symptoms require urgent medical attention?"

results = {
    "k=3": run_dense_search(query, vectorstore, k=3),
    "k=5": run_dense_search(query, vectorstore, k=5),
    "k=7": run_dense_search(query, vectorstore, k=7),
}

for k_label, docs in results.items():
    print("\n" + "="*80)
    print(f"Results for {k_label}")
    print("="*80)

    for i, doc in enumerate(docs, 1):
        meta = doc.metadata

        print(f"\nResult {i}")
        print("Global ID :", meta.get("global_chunk_id"))
        print("Chunk #   :", meta.get("chunk_index"))
        print("Topic     :", meta.get("topic"))
        print("Section   :", meta.get("section_title"))
        print("Source    :", meta.get("source"))
        print("Author    :", meta.get("author"))
        print("Date      :", meta.get("document_date"))
        print("URL       :", meta.get("url"))
        print("Page #    :", meta.get("page_number"))
        print("Text      :", doc.page_content[:300])  # preview


Results for k=3

Result 1
Global ID : 134
Chunk #   : 2
Topic     : Allergic Reaction (Anaphylaxis)
Section   : Symptoms
Source    : MedlinePlus
Author    : None
Date      : 2024-03-31
URL       : https://medlineplus.gov/ency/article/000844.htm
Page #    : None
Text      : Symptoms develop quickly, often within seconds or minutes. They may include any of the following: Abdominal pain Feeling anxious Chest discomfort or tightness Diarrhea Difficulty breathing, coughing, wheezing, or high-pitched breathing sounds Difficulty swallowing Dizziness or lightheadedness Hives 

Result 2
Global ID : 364
Chunk #   : 14
Topic     : Insect Bites and Stings
Section   : Immediate action required: Call 999 if:
Source    : NHS
Author    : None
Date      : 2023-06-01
URL       : https://www.nhs.uk/conditions/insect-bites-and-stings/
Page #    : None
Text      : your lips, mouth, throat or tongue suddenly become swollen you're breathing very fast or struggling to breathe (you may become very wheezy or f

BM25 retriever

In [28]:
!pip install rank_bm25


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# =========================================
# 2) Sparse retrieval with BM25
# =========================================

# install first if needed:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunked_docs)
print("BM25 retriever created.")

def get_bm25_retriever(documents, k=5):
    retriever = BM25Retriever.from_documents(documents)
    retriever.k = k
    return retriever


def run_bm25_search(query, documents, k=5):
    retriever = get_bm25_retriever(documents, k=k)
    docs = retriever.invoke(query)
    return docs

BM25 retriever created.


In [30]:
# example usage 
query = "What symptoms require urgent medical attention?"

bm25_results = {
    "k=3": run_bm25_search(query, chunked_docs, k=3),
    "k=5": run_bm25_search(query, chunked_docs, k=5),
    "k=7": run_bm25_search(query, chunked_docs, k=7),
}

for k_label, docs in bm25_results.items():
    print("\n" + "="*80)
    print(f"BM25 Results for {k_label}")
    print("="*80)

    for i, doc in enumerate(docs, 1):
        meta = doc.metadata

        print(f"\nResult {i}")
        print("Global ID :", meta.get("global_chunk_id"))
        print("Chunk #   :", meta.get("chunk_index"))
        print("Topic     :", meta.get("topic"))
        print("Section   :", meta.get("section_title"))
        print("Source    :", meta.get("source"))
        print("Author    :", meta.get("author"))
        print("Date      :", meta.get("document_date"))
        print("URL       :", meta.get("url"))
        print("Page #    :", meta.get("page_number"))
        print("Text      :", doc.page_content[:300])


BM25 Results for k=3

Result 1
Global ID : 304
Chunk #   : 7
Topic     : Sunburn
Section   : What to Expect at Your Office Visit
Source    : MedlinePlus
Author    : None
Date      : 2025-02-11
URL       : https://medlineplus.gov/ency/article/003227.htm
Page #    : None
Text      : Your provider will perform a physical exam and look at your skin. You may be asked about your medical history and current symptoms, including: When did the sunburn occur? How often do you get sunburn? Do you have blisters? How much of the body was sunburned? What medicines do you take? Do you use a 

Result 2
Global ID : 11
Chunk #   : 11
Topic     : Chest Pain
Section   : What to Expect at Your Office Visit
Source    : MedlinePlus
Author    : None
Date      : 2024-05-08
URL       : https://medlineplus.gov/ency/article/003079.htm
Page #    : None
Text      : pain better after you take nitroglycerin medicine? After you eat or take antacids? After you belch? What other symptoms do you have? The types of tests 

# Hybrid

In [31]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

# =========================================
# Text preprocessing for BM25
# =========================================
def bm25_preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()


# =========================================
# Cosine similarity
# =========================================
def cosine_similarity(a, b):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)

    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    if a_norm == 0 or b_norm == 0:
        return 0.0

    return float(np.dot(a, b) / (a_norm * b_norm))


# =========================================
# Full-corpus hybrid retrieval
# =========================================
def run_full_hybrid_search(query, chunked_docs, embedding_model, dense_weight=0.7, bm25_weight=0.3):
    """
    Computes dense + BM25 scores over the FULL dataset,
    combines them, and returns one fixed global ranking.
    """

    # -------------------------------------
    # 1) Dense scores over full corpus
    # -------------------------------------
    query_emb = embedding_model.embed_query(query)
    doc_texts = [doc.page_content for doc in chunked_docs]
    doc_embs = embedding_model.embed_documents(doc_texts)

    dense_scores = [
        cosine_similarity(query_emb, doc_emb)
        for doc_emb in doc_embs
    ]

    # -------------------------------------
    # 2) BM25 scores over full corpus
    # -------------------------------------
    tokenized_corpus = [bm25_preprocess(doc.page_content) for doc in chunked_docs]
    bm25 = BM25Okapi(tokenized_corpus)

    tokenized_query = bm25_preprocess(query)
    bm25_scores = bm25.get_scores(tokenized_query)  # one score per doc

    # -------------------------------------
    # 3) Normalize both score lists
    # -------------------------------------
    dense_scores = np.array(dense_scores, dtype=np.float32)
    bm25_scores = np.array(bm25_scores, dtype=np.float32)

    # Min-max normalization
    if dense_scores.max() > dense_scores.min():
        dense_scores_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min())
    else:
        dense_scores_norm = np.zeros_like(dense_scores)

    if bm25_scores.max() > bm25_scores.min():
        bm25_scores_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
    else:
        bm25_scores_norm = np.zeros_like(bm25_scores)

    # -------------------------------------
    # 4) Weighted hybrid score
    # -------------------------------------
    hybrid_scores = dense_weight * dense_scores_norm + bm25_weight * bm25_scores_norm

    # -------------------------------------
    # 5) Build ONE fixed final ranking
    # -------------------------------------
    ranked_indices = np.argsort(-hybrid_scores)  # descending

    ranked_docs = [chunked_docs[i] for i in ranked_indices]
    ranked_scores = [float(hybrid_scores[i]) for i in ranked_indices]
    ranked_dense_scores = [float(dense_scores_norm[i]) for i in ranked_indices]
    ranked_bm25_scores = [float(bm25_scores_norm[i]) for i in ranked_indices]

    return ranked_docs, ranked_scores, ranked_dense_scores, ranked_bm25_scores

In [32]:
query = "What symptoms require urgent medical attention?"

all_docs, all_scores, all_dense, all_bm25 = run_full_hybrid_search(
    query=query,
    chunked_docs=chunked_docs,
    embedding_model=embedding_model,
    dense_weight=0.7,
    bm25_weight=0.3
)

In [33]:
hybrid_results = {
    "k=3": (all_docs[:3], all_scores[:3], all_dense[:3], all_bm25[:3]),
    "k=5": (all_docs[:5], all_scores[:5], all_dense[:5], all_bm25[:5]),
    "k=7": (all_docs[:7], all_scores[:7], all_dense[:7], all_bm25[:7]),
}

for k_label, (docs, scores, dense_s, bm25_s) in hybrid_results.items():
    print("\n" + "="*80)
    print(f"FULL-CORPUS HYBRID Results for {k_label}")
    print("="*80)

    for i, (doc, score, d_score, b_score) in enumerate(zip(docs, scores, dense_s, bm25_s), 1):
        meta = doc.metadata

        print(f"\nResult {i}")
        print("Hybrid score:", round(score, 6))
        print("Dense score :", round(d_score, 6))
        print("BM25 score  :", round(b_score, 6))
        print("Global ID   :", meta.get("global_chunk_id"))
        print("Chunk #     :", meta.get("chunk_index"))
        print("Topic       :", meta.get("topic"))

        print("Section     :", meta.get("section_title"))
        print("Source      :", meta.get("source"))
        print("Author      :", meta.get("author"))
        print("Date        :", meta.get("document_date"))
        print("URL         :", meta.get("url"))
        print("Page #      :", meta.get("page_number"))
        print("Text        :", doc.page_content[:250])


FULL-CORPUS HYBRID Results for k=3

Result 1
Hybrid score: 0.744388
Dense score : 0.779581
BM25 score  : 0.662271
Global ID   : 190
Chunk #     : 3
Topic       : Meningitis
Section     : When to get medical help
Source      : NHS
Author      : None
Date        : 2026-03-18
URL         : https://www.nhs.uk/conditions/meningitis/
Page #      : None
Text        : Call 999 for an ambulance or go to your nearest A&E immediately if you think you or someone you look after could have meningitis or sepsis. Trust your instincts and do not wait for all the symptoms to appear or until a rash develops. Someone with men

Result 2
Hybrid score: 0.742008
Dense score : 1.0
BM25 score  : 0.140027
Global ID   : 134
Chunk #     : 2
Topic       : Allergic Reaction (Anaphylaxis)
Section     : Symptoms
Source      : MedlinePlus
Author      : None
Date        : 2024-03-31
URL         : https://medlineplus.gov/ency/article/000844.htm
Page #      : None
Text        : Symptoms develop quickly, often within seco

# Citation

In [34]:
def build_context_and_ieee_sources(question, vectorstore, k=5):
    """
    Retrieve top-k chunks and format:
    - CONTEXT: only numbered chunks (clean)
    - SOURCES: full IEEE-style references
    """

    retrieved_docs = run_dense_search(question, vectorstore, k=k)

    context_blocks = []
    source_lines = []

    for i, doc in enumerate(retrieved_docs, start=1):
        meta = doc.metadata

        topic = meta.get("topic", "Unknown Topic")
        source = meta.get("source", "Unknown Source")
        url = meta.get("url", "No URL")
        section_title = meta.get("section_title", "Unknown Section")
        document_date = meta.get("document_date", "n.d.")

        context_block = f"[{i}]\n{doc.page_content}"
        context_blocks.append(context_block)

        source_line = (
            f'[{i}] {source}, "{topic}," '
            f'section: "{section_title}," {document_date}. '
            f'[Online]. Available: {url}'
        )
        source_lines.append(source_line)

    context_text = "\n\n".join(context_blocks)
    sources_text = "\n".join(source_lines)

    return context_text, sources_text, retrieved_docs

In [35]:
question = "My 2-month-old infant has a fever of 38.8°C and is unusually sleepy."

context_text, sources_text, retrieved_docs = build_context_and_ieee_sources(
    question=question,
    vectorstore=vectorstore,
    k=3
)

print("========== CONTEXT ==========\n")
print(context_text[:5000])

print("\n========== SOURCES ==========\n")
print(sources_text)

========== CONTEXT ==========

[1]
is under 3 months old and has a temperature of 38C or higher, or you think they have a high temperature is 3 to 6 months old and has a temperature of 39C or higher, or you think they have a high temperature has other signs of illness, such as

[2]
A normal temperature in babies and children can vary slightly from child to child. A high temperature is 38C or more. If your child has a high temperature, they might: feel hotter than usual when you touch their back or chest feel sweaty look or feel unwell have a seizure or fit, called a febrile seizure Use a digital thermometer, which you can buy from pharmacies and supermarkets, to take your child's temperature. Place the thermometer inside the top of the armpit. Gently close the arm over the thermometer and keep it pressed to the side of the body. Leave the thermometer in place for as long as it says in the instruction leaflet. Some digital thermometers beep when they're ready. Remove the thermometer. Th

# Prompt Template

LLama model

In [36]:
import os
from openai import OpenAI

LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")
#print(LLAMA_API_KEY)

client_or = OpenAI(
    api_key=LLAMA_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [37]:
import os
key = os.getenv("LLAMA_API_KEY")
print("Loaded:", key is not None)
print("Prefix:", key[:12] if key else None)

Loaded: True
Prefix: sk-or-v1-70f


In [38]:
def generate_with_llama(prompt, model_name="meta-llama/llama-3-8b-instruct"):
    response = client_or.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=500
    )
    return response.choices[0].message.content

  
Runs dense retrieval and also returns similarity scores.
Lower score usually means more similar for Chroma distance-based search.
   

In [39]:
def run_dense_search_with_scores(query, vectorstore, k=3):

    results = vectorstore.similarity_search_with_score(query, k=k)
    return results

References

In [40]:
def build_context_and_ieee_sources(question, vectorstore, k=3):
    """
    Retrieve top-k chunks using dense retrieval and format:
    - CONTEXT: numbered chunks only
    - SOURCES: full IEEE-style references
    - retrieved_docs
    - scored_results
    """
    scored_results = run_dense_search_with_scores(question, vectorstore, k=k)
    retrieved_docs = [doc for doc, score in scored_results]

    context_blocks = []
    source_lines = []

    for i, doc in enumerate(retrieved_docs, start=1):
        meta = doc.metadata

        topic = meta.get("topic", "Unknown Topic")
        source = meta.get("source", "Unknown Source")
        url = meta.get("url", "No URL")
        section_title = meta.get("section_title", "Unknown Section")
        document_date = meta.get("document_date", "n.d.")

        context_block = f"[{i}]\n{doc.page_content}"
        context_blocks.append(context_block)

        source_line = (
            f'[{i}] {source}, "{topic}," '
            f'section: "{section_title}," {document_date}. '
            f'[Online]. Available: {url}'
        )
        source_lines.append(source_line)

    context_text = "\n\n".join(context_blocks)
    sources_text = "\n".join(source_lines)

    return context_text, sources_text, retrieved_docs, scored_results

Fall back detection

In [41]:
def has_sufficient_context_scored(scored_results, max_best_score=1.2, min_docs=1):
    """
    Decide whether retrieval is strong enough for grounded generation.
    Lower score = better match in Chroma.
    """

    if not scored_results or len(scored_results) < min_docs:
        return False

    best_score = scored_results[0][1]

    if best_score > max_best_score:
        return False

    return True

prompt template

In [42]:
def build_grounded_prompt(question, context_text, sources_text):
    """
    Final grounded prompt for ER nurse triage with FULL IEEE references in output.
    """

    prompt = f"""
You are an Emergency Department triage assistant supporting an ER nurse.

## Purpose
Help the nurse determine the appropriate level of care based ONLY on the provided clinical context.

## Critical Rules
- Use ONLY the provided context.
- Do NOT use outside medical knowledge.
- Do NOT provide a diagnosis.
- If the context is insufficient, say exactly:
"I don't have enough information about this in my knowledge base."

## Urgency Categories (choose EXACTLY ONE)
- Urgent: Patient requires immediate or same-day evaluation in the ER (consider admission or urgent workup).
- Routine: Patient can be managed with non-urgent evaluation (clinic or scheduled appointment).
- Self-care: Patient can be safely managed at home with monitoring and basic care.

## Clinical Framing
- Assume the user is an ER nurse making triage decisions.
- Focus on identifying red flags and risk features.
- Be conservative when serious symptoms are present.
- Suggest whether the patient should:
  - remain in ER / be admitted
  - receive urgent evaluation
  - be discharged with follow-up
  - be managed at home

## Citation Rules
- Use citation numbers like [1], [2], [3] inline in the Reasoning and Next steps.
- Only cite sources provided below.
- Do NOT invent sources.
- At the end, include the FULL IEEE-style references corresponding to the citations used.

## CONTEXT:
{context_text}

## AVAILABLE SOURCES (IEEE FORMAT):
{sources_text}

## USER QUESTION:
{question}

## Output Format (STRICT)

Urgency: <Urgent / Routine / Self-care / Insufficient information>

Reasoning:
<2–4 concise sentences based ONLY on the provided context. Include inline citations like [1], [2].>

Recommendation:
<Clear clinical recommendation: admit / urgent evaluation / outpatient follow-up / home care.>

Next steps:
- <Bullet point 1 with citations if relevant>
- <Bullet point 2 with citations if relevant>
- <Bullet point 3 (optional)>

Sources:
<List the FULL IEEE-style references corresponding ONLY to the citation numbers used>
"""
    return prompt

Fallback prompt template

In [43]:
def build_fallback_judgment_prompt(question):
    """
    Used when retrieval is not strong enough for grounded generation.
    """
    prompt = f"""
You are an Emergency Department triage assistant supporting an ER nurse.

The retrieved knowledge base did not provide enough reliable context for a grounded answer.

Your task is to provide a cautious BEST-EFFORT triage judgment based ONLY on the patient symptom description below.

## Critical Rules
- Choose EXACTLY ONE:
  - Urgent
  - Routine
  - Self-care
- Do NOT give a diagnosis.
- Do NOT prescribe medication.
- Be conservative when symptoms sound potentially serious.
- Include EXACTLY this sentence in your reasoning:
"I don't have enough information about this in my knowledge base."
- Clearly state that this is a provisional judgment based only on the reported symptoms.
- Do NOT include citations.
- Sources must be None.

## USER QUESTION:
{question}

## Output Format (STRICT)

Urgency: <Urgent / Routine / Self-care>

Reasoning:
<1–3 concise sentences INCLUDING EXACTLY this sentence: "I don't have enough information about this in my knowledge base.">

Recommendation:
<Short clinical recommendation for review>

Next steps:
- <Bullet 1>
- <Bullet 2>
- <Bullet 3 if needed>

Sources:
None
"""
    return prompt.strip()

preparing the input for rag

In [44]:
def prepare_rag_input(question, vectorstore, k=3):
    context_text, sources_text, retrieved_docs, scored_results = build_context_and_ieee_sources(
        question=question,
        vectorstore=vectorstore,
        k=k
    )

    enough_context = has_sufficient_context_scored(
        scored_results=scored_results,
        max_best_score=1.2,
        min_docs=1
    )

    if not enough_context:
        return {
            "question": question,
            "retrieved_docs": retrieved_docs,
            "scored_results": scored_results,
            "context_text": context_text,
            "sources_text": sources_text,
            "prompt": None,
            "has_context": False
        }

    final_prompt = build_grounded_prompt(
        question=question,
        context_text=context_text,
        sources_text=sources_text
    )

    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "scored_results": scored_results,
        "context_text": context_text,
        "sources_text": sources_text,
        "prompt": final_prompt,
        "has_context": True
    }

In-text citation

In [45]:

import re

def extract_source_map(sources_text):
    source_map = {}
    for line in sources_text.splitlines():
        match = re.match(r'^\[(\d+)\]\s*(.+)$', line.strip())
        if match:
            source_map[int(match.group(1))] = match.group(2)
    return source_map


def normalize_citations_and_sources(answer, sources_text):
    source_map = extract_source_map(sources_text)

    if "\nSources:" in answer:
        main_text = answer.split("\nSources:")[0].rstrip()
    else:
        main_text = answer.rstrip()

    citation_pattern = re.compile(r'\[(\d+)\]')
    original_citations = [int(x) for x in citation_pattern.findall(main_text)]

    ordered_unique = []
    for c in original_citations:
        if c in source_map and c not in ordered_unique:
            ordered_unique.append(c)

    remap = {old: new for new, old in enumerate(ordered_unique, start=1)}

    def replace_citation(match):
        old_num = int(match.group(1))
        if old_num in remap:
            return f'[{remap[old_num]}]'
        return ''

    normalized_text = citation_pattern.sub(replace_citation, main_text)
    normalized_text = re.sub(r'\s+\.', '.', normalized_text)
    normalized_text = re.sub(r'\(\s*\)', '', normalized_text)
    normalized_text = re.sub(r'(?m)[ \t]+$', '', normalized_text)
    normalized_text = re.sub(r'\n{3,}', '\n\n', normalized_text).strip()

    used_sources = []
    for old_num in ordered_unique:
        new_num = remap[old_num]
        used_sources.append(f'[{new_num}] {source_map[old_num]}')

    if used_sources:
        return normalized_text + "\n\nSources:\n" + "\n".join(used_sources)
    return normalized_text + "\n\nSources:\nNone"

def answer_question_with_rag_llama(question, vectorstore, k=3):
    rag_input = prepare_rag_input(question, vectorstore, k=k)

    # Mode 1: grounded RAG
    if rag_input["has_context"]:
        final_answer = generate_with_llama(rag_input["prompt"])
        final_answer = normalize_citations_and_sources(
            final_answer,
            rag_input["sources_text"]
        )
        rag_input["mode"] = "grounded_rag"
        return final_answer, rag_input

    # Mode 2: fallback judgment
    fallback_prompt = build_fallback_judgment_prompt(question)
    final_answer = generate_with_llama(fallback_prompt)
    rag_input["mode"] = "fallback_judgment"
    rag_input["fallback_prompt"] = fallback_prompt
    return final_answer, rag_input

Example usage

In [46]:
question = "My 2-month-old infant has a fever of 38.8°C and is unusually sleepy."

rag_input = prepare_rag_input(question, vectorstore, k=3)

if rag_input["has_context"]:
    print("MODE: grounded_rag\n")
    print(rag_input["prompt"])
else:
    print("MODE: fallback_judgment\n")
    print(build_fallback_judgment_prompt(question))

MODE: grounded_rag


You are an Emergency Department triage assistant supporting an ER nurse.

## Purpose
Help the nurse determine the appropriate level of care based ONLY on the provided clinical context.

## Critical Rules
- Use ONLY the provided context.
- Do NOT use outside medical knowledge.
- Do NOT provide a diagnosis.
- If the context is insufficient, say exactly:
"I don't have enough information about this in my knowledge base."

## Urgency Categories (choose EXACTLY ONE)
- Urgent: Patient requires immediate or same-day evaluation in the ER (consider admission or urgent workup).
- Routine: Patient can be managed with non-urgent evaluation (clinic or scheduled appointment).
- Self-care: Patient can be safely managed at home with monitoring and basic care.

## Clinical Framing
- Assume the user is an ER nurse making triage decisions.
- Focus on identifying red flags and risk features.
- Be conservative when serious symptoms are present.
- Suggest whether the patient should:
  - 

In [47]:
question = "My 2-month-old infant has a fever of 38.8°C and is unusually sleepy."

answer, rag_input = answer_question_with_rag_llama(
    question=question,
    vectorstore=vectorstore,
    k=3
)

print("MODE:", rag_input["mode"])
print("\n========== FINAL ANSWER ==========\n")
print(answer)

MODE: grounded_rag

========== FINAL ANSWER ==========

Urgency: Urgent

Reasoning:
This 2-month-old infant has a fever of 38.8°C, which is above the normal temperature range for this age group [1]. Additionally, the infant is unusually sleepy, which could be a sign of illness [1]. A high temperature in infants under 3 months old requires immediate evaluation in the ER [1].

Recommendation:
Admit to the ER for further evaluation and management.

Next steps:
• Obtain a complete medical history and perform a physical examination to identify any other signs of illness.
• Consider performing a sepsis workup, including blood cultures and a complete blood count, to rule out serious infections [1].
• Provide supportive care, such as oxygen therapy and fluid management, as needed.

Sources:
[1] NHS, "Fever in Infants," section: "Urgent advice: Call 111 or your GP surgery now if your child:," 2024-01-03. [Online]. Available: https://www.nhs.uk/conditions/fever-in-children/


In [48]:
question = "i have a headache"
answer, rag_input = answer_question_with_rag_llama(
    question=question,
    vectorstore=vectorstore,
    k=3
)

print("MODE:", rag_input["mode"])
print("\n========== FINAL ANSWER ==========\n")
print(answer)

MODE: grounded_rag

========== FINAL ANSWER ==========

Urgency: Routine

Reasoning:
The provided context describes a headache, which is a common symptom of migraines. According to [1], migraines are a recurring type of headache that can cause moderate to severe pain. The context does not mention any red flags or risk features that would indicate a more urgent evaluation is necessary.

Recommendation:
The patient can be managed with non-urgent evaluation in a clinic or with a scheduled appointment.

Next steps:
• The patient should be referred to a primary care physician or a neurologist for further evaluation and management.
• The patient should be advised to keep a headache diary to track the frequency, duration, and severity of their headaches, as well as any potential triggers [2].

Sources:
[1] MedlinePlus, "Migraine," section: "What are migraines?," 2025-11-20. [Online]. Available: https://medlineplus.gov/migraine.html
[2] MedlinePlus, "Migraine," section: "What are the symptoms 

In [49]:
question = "i think i have breast cancer, i have a bump"
answer, rag_input = answer_question_with_rag_llama(
    question=question,
    vectorstore=vectorstore,
    k=3
)

print("MODE:", rag_input["mode"])
print("\n========== FINAL ANSWER ==========\n")
print(answer)

MODE: grounded_rag

========== FINAL ANSWER ==========

Urgency: Routine

Reasoning:
The patient is concerned about a bump on their breast, which they suspect may be breast cancer. However, without further information, it is not possible to determine the urgency of the situation. The patient should be evaluated by a healthcare provider, but it is not an emergency. [1]

Recommendation:
The patient should be evaluated by a healthcare provider, but it is not an emergency. They should be scheduled for an outpatient appointment.

Next steps:
• The patient should schedule an appointment with their primary care physician or a dermatologist to have the bump evaluated.
• The patient should be prepared to provide a detailed description of the bump, including its size, shape, color, and any changes it has undergone.

Sources:
[1] NHS, "Warts and Verrucas," section: "Non-urgent advice: See a GP if:," 2023-07-25. [Online]. Available: https://www.nhs.uk/conditions/warts-and-verrucas/
